In [52]:
# %pip install pandas geopandas requests
import pandas as pd
import geopandas as gpd
import requests

In [53]:
## import dataset and clean 
gdf = gpd.read_file('../amenities_with_importance_score.geojson')

print("Columns in the GeoDataFrame:")
print(gdf.columns.tolist())

columns_to_keep = ['amenity_id', 'amenity_type', 'planning_area']

# Keep only the specified columns
gdf = gdf[columns_to_keep]

# change planning area to string
gdf['planning_area'] = gdf['planning_area'].astype(str)

gdf['footfall'] = 0.0  # Initialize footfall column

print("Initial GeoDataFrame:")
print(gdf.head())

print("check type of each column:")
print(gdf.dtypes)

Columns in the GeoDataFrame:
['amenity_id', 'amenity_type', 'amenity_name', 'road_name', 'postal_code', 'geom_type', 'lon', 'lat', 'source_file', 'ADDRESSBUILDINGNAME', 'ADDRESSBLOCKHOUSENUMBER', 'LANDXADDRESSPOINT', 'LANDYADDRESSPOINT', 'PRECINCTPA', 'Shape__Area', 'Shape__Length', 'geometry_wkt', 'geometry_geojson', 'amenity_category', 'amenity_priority', 'amenity_weight', 'importance_score', 'importance_label', 'planning_area', 'subzone', 'geometry']
Initial GeoDataFrame:
                             amenity_id amenity_type planning_area  footfall
0  8d39567d-5ea6-5751-beec-88770a9be781   bus_depots        BISHAN       0.0
1  fbed27ca-8066-5e40-8cc5-d62cadfb738c   bus_depots    ANG MO KIO       0.0
2  fb09f377-194b-5c85-9942-cc74e9136575   bus_depots    ANG MO KIO       0.0
3  187dda5f-1316-5f06-854b-0718ea11f55c   bus_depots       HOUGANG       0.0
4  eb89bf2e-7583-5961-95b3-2c647513f432   bus_depots      TAMPINES       0.0
check type of each column:
amenity_id        object
amenit

In [54]:
#mark if planning area is heartland
non_heartlands = ['OUTRAM', 'MUSEUM', 'NEWTON', 'RIVER VALLEY', 'SINGAPORE RIVER', 'MARINA SOUTH', 'MARINA EAST', 'STRAITS VIEW', 'ROCHOR', 'ORCHARD', 'DOWNTOWN CORE']
for index, row in gdf.iterrows():
    gdf.at[index, 'is_heartland'] = not (row['planning_area'] in non_heartlands)
print("GeoDataFrame with heartland labels:")
print(gdf.head(10))

GeoDataFrame with heartland labels:
                             amenity_id amenity_type planning_area  footfall  \
0  8d39567d-5ea6-5751-beec-88770a9be781   bus_depots        BISHAN       0.0   
1  fbed27ca-8066-5e40-8cc5-d62cadfb738c   bus_depots    ANG MO KIO       0.0   
2  fb09f377-194b-5c85-9942-cc74e9136575   bus_depots    ANG MO KIO       0.0   
3  187dda5f-1316-5f06-854b-0718ea11f55c   bus_depots       HOUGANG       0.0   
4  eb89bf2e-7583-5961-95b3-2c647513f432   bus_depots      TAMPINES       0.0   
5  9f90d676-aa68-561b-b7df-d4197af2cfec   bus_depots   BUKIT BATOK       0.0   
6  c2d042e1-f588-5613-94c6-c8b8df7b97e2   bus_depots     TOA PAYOH       0.0   
7  b4a4336a-616a-51ff-84a1-38c798ea10b6   bus_depots   JURONG WEST       0.0   
8  5ce8041a-bab4-5e2f-9b5f-2270b24bdac4   bus_depots    QUEENSTOWN       0.0   
9  5f7c06af-f531-5c81-87af-44da616a0ee8   bus_depots       HOUGANG       0.0   

  is_heartland  
0         True  
1         True  
2         True  
3         True 

In [55]:
## this is for the population by planing area
## load subzone population (from census 2020 data.gov.sg)
dataset_id = "d_e7ae90176a68945837ad67892b898466"
API_url = "https://data.gov.sg/api/action/datastore_search?resource_id="  + dataset_id

response = requests.get(API_url)
print(response.json())
data = response.json()
# extract records from API response
if 'result' in data and 'records' in data['result']:
    planning_area_records = data['result']['records']
    
    # Create DataFrame from the records
    planning_area_population = pd.DataFrame(planning_area_records)
    planning_area_population = planning_area_population[['Number', 'Total_Total']]  # Keep only relevant columns
    
    print("Planning area population DataFrame:")
    print(planning_area_population.head(10))

# keep only population of planning areas 
# Keep all rows where Number ends with "- Total"
population_by_planning_area_df = planning_area_population.copy()

# Clean up the planning area names by removing "- Total"
population_by_planning_area_df['planning_area'] = population_by_planning_area_df['Number'].str.replace('- Total', '')

#remove leading and trailing spaces
population_by_planning_area_df['planning_area'] = population_by_planning_area_df['planning_area'].str.strip()

# drop the original 'Number' column as it's no longer needed
population_by_planning_area_df.drop(columns=['Number'], inplace=True)

# Convert Total_Total to numeric, forcing errors to NaN
population_by_planning_area_df['Total_Total'] = pd.to_numeric(population_by_planning_area_df['Total_Total'], errors='coerce')

#change all to upper case
population_by_planning_area_df['planning_area'] = population_by_planning_area_df['planning_area'].str.upper()

print("Planning area DataFrame with cleaned names:")
print(population_by_planning_area_df.shape)
print(population_by_planning_area_df.head(100))

# find jurong west planning area
jurong_west_population = population_by_planning_area_df.loc[
    population_by_planning_area_df['planning_area'] == 'JURONG WEST', 'Total_Total'
].values
print(f"Population of Jurong West: {jurong_west_population}")


{'help': 'https://data.gov.sg/api/3/action/help_show?name=datastore_search', 'success': True, 'result': {'resource_id': 'd_e7ae90176a68945837ad67892b898466', 'fields': [{'type': 'text', 'id': 'Number'}, {'type': 'text', 'id': 'Total_Total'}, {'type': 'text', 'id': 'Total_Males'}, {'type': 'text', 'id': 'Total_Females'}, {'type': 'text', 'id': 'Chinese_Total'}, {'type': 'text', 'id': 'Chinese_Males'}, {'type': 'text', 'id': 'Chinese_Females'}, {'type': 'text', 'id': 'Malays_Total'}, {'type': 'text', 'id': 'Malays_Males'}, {'type': 'text', 'id': 'Malays_Females'}, {'type': 'text', 'id': 'Indians_Total'}, {'type': 'text', 'id': 'Indians_Males'}, {'type': 'text', 'id': 'Indians_Females'}, {'type': 'text', 'id': 'Others_Total'}, {'type': 'text', 'id': 'Others_Males'}, {'type': 'text', 'id': 'Others_Females'}, {'type': 'int4', 'id': '_id'}], 'records': [{'_id': 1, 'Number': 'Total', 'Total_Total': '4044210', 'Total_Males': '1977560', 'Total_Females': '2066650', 'Chinese_Total': '3006770', 'C

In [56]:
## split amenities into different types to be accounted for
neighbourhood_amenities = ['bus_depots', 'bus_interchanges_terminals', 'bus_stops', 'childcare_clean',
 'community_clubs', 'fire_services', 'hdb_buildings', 'hdb_points_shp', 'libraries',
 'mrt_station_exits', 'other_institutions',
 'parkfacilities', 'police', 'post_offices',
 'sports_centres', 'stadiums', 'swimming_complex'] # these will be divided by total population / number per planning area

by_age_amenities = ['kindergartens', 'moe_schools','preschools', 'higher_education'] # these proxied by age cat, then divided by total number 

by_demographic = 'special_education' # proxied by special education enrolment numbers

religious_amenities = ['chinese_temples', 'churches', 'indian_temples', 'mosques', 'synagogues', 'sikh_temples'] #these will be divided by religious demographic / total number 

country_wide_amenities = ['courts','concert_halls']

tourism_amenities = ['hotels' , 'tourist_attractions', 'historic_sites'] #these are using tourism numbers

In [57]:
# amenity count by planning area column 1 (amenity type) 2 (planning area) 3 (count)
amenity_count_by_planning_area = gdf.groupby(['amenity_type', 'planning_area']).size().reset_index(name='count')
print("Amenity count by planning area:")
print(amenity_count_by_planning_area.head(10))

amenity_count_whole_country = gdf.groupby(['amenity_type']).size().reset_index(name='count')
print("Amenity count for whole country:")
print(amenity_count_whole_country.head(10))

Amenity count by planning area:
  amenity_type planning_area  count
0   bus_depots    ANG MO KIO      2
1   bus_depots        BISHAN      1
2   bus_depots   BUKIT BATOK      1
3   bus_depots       HOUGANG      2
4   bus_depots   JURONG WEST      1
5   bus_depots    QUEENSTOWN      1
6   bus_depots  SUNGEI KADUT      2
7   bus_depots      TAMPINES      2
8   bus_depots     TOA PAYOH      1
9   bus_depots     WOODLANDS      1
Amenity count for whole country:
                 amenity_type  count
0                  bus_depots     14
1  bus_interchanges_terminals     44
2                   bus_stops   4882
3             childcare_clean   1545
4             chinese_temples    302
5                    churches    391
6             community_clubs    117
7               concert_halls     27
8                      courts     15
9               fire_services     56


In [ ]:
total_population = 6110000  # https://www.population.gov.sg/our-population/population-trends/overall-population/
frequency_neighbourhood = 7
frequency_special_needs = 5
frequency_religious = 1
frequency_age = 5 
frequency_country_wide = 0.02 # 1x a year, 1/52 weeks


for idx, row in gdf.iterrows():
    amenity_type = row['amenity_type']

    # --- Neighbourhood amenities ---
    if amenity_type in neighbourhood_amenities:
        # get planning area
        planning_area = row['planning_area']
        print(f"Processing {amenity_type} in {planning_area}")
        
        # use planning area to find population in planning area - ADD ERROR HANDLING
        population_matches = population_by_planning_area_df.loc[
            population_by_planning_area_df['planning_area'] == planning_area, 'Total_Total'
        ].values
        
        # if dont have, divide population by number of that amenity type in whole country
        if len(population_matches) == 0:
            print(f"Warning: No population data found for planning area '{planning_area}'. Using average population.")
            country_amenity_count = amenity_count_whole_country.loc[
                amenity_count_whole_country['amenity_type'] == amenity_type, 'count'
            ].values
            population_in_planning_area = total_population / 55 #this is average population per planning area
        
        else:
            population_of_planning_area = population_matches[0]
        
        # get number of amenities of that type in that planning area - ADD ERROR HANDLING
        amenity_matches = amenity_count_by_planning_area.loc[
            (amenity_count_by_planning_area['amenity_type'] == amenity_type) & 
            (amenity_count_by_planning_area['planning_area'] == planning_area), 'count'
        ].values
        
        if len(amenity_matches) == 0:
            print(f"Warning: No amenity count found for {amenity_type} in {planning_area}. Skipping.")
            continue
            
        amenity_count = amenity_matches[0]
        
        # calculate footfall
        if amenity_count > 0:
            footfall = (population_of_planning_area / amenity_count) * frequency_neighbourhood
            gdf.at[idx, 'footfall'] = footfall
            print(f"Set footfall to {footfall} for {amenity_type} in {planning_area}")
        else:
            print(f"Warning: Amenity count is 0 for {amenity_type} in {planning_area}")

    # ---  Age-based amenities ---
    elif amenity_type in by_age_amenities:
        age_demographic = { #https://www.statista.com/statistics/624913/singapore-population-by-age-group/
                    'kindergartens': (1705000/5)*2 , # 5-6 years
                    'moe_schools': (2020800/5*3)+2043900+(2110700/5*4), # 7-18 years
                    'preschools': (1705000/5*2), # 3-4 years
                    'higher_education': 2246400*0.8,  #assume 80% people aged 20-24 years old pursue higher education
                    'childcare_clean': round(1705000/5*4*0.9, 0) # 90% of children aged 3-6 are in childcare
                }
        # get number of amenities of that type in whole country - ADD ERROR HANDLING
        amenity_matches = amenity_count_whole_country.loc[
            amenity_count_whole_country['amenity_type'] == amenity_type, 'count'
        ].values

        #divide age demographic by number of amenities
        if len(amenity_matches) == 0:
            print(f"Warning: No amenity count found for age-based amenity {amenity_type}. Skipping.")
            continue
        amenity_count = amenity_matches[0]
        population_for_age_group = age_demographic[amenity_type]
        footfall = (population_for_age_group / amenity_count) * frequency_age
        gdf.at[idx, 'footfall'] = footfall
        print(f"Set footfall to {footfall} for age-based amenity {amenity_type}")
        

    # --- Religious amenities ---
    elif amenity_type in religious_amenities:
        population_percentage_by_religion = { #https://www.singstat.gov.sg/-/media/files/visualising_data/infographics/c2020/c2020-religion.ashx
            'chinese_temples': round(0.311 + 0.088, 2),
            'churches': 0.19,
            'indian_temples': 0.15,
            'mosques': 0.05,
            'synagogues': 0.01,
            'sikh_temples': 0.01
        }

        population_by_religion = {
            religion: total_population * pct
            for religion, pct in population_percentage_by_religion.items()
        }

        # ADD ERROR HANDLING for religious amenities 
        religion_matches = amenity_count_whole_country.loc[
            amenity_count_whole_country['amenity_type'] == amenity_type, 'count'
        ].values
        
        if len(religion_matches) == 0:
            print(f"Warning: No count data found for religious amenity {amenity_type}. Skipping.")
            continue
            
        amenity_count = religion_matches[0]
        population_for_religion = population_by_religion[amenity_type]
        footfall = (population_for_religion / amenity_count) * frequency_religious
        
        gdf.at[idx, 'footfall'] = footfall
        print(f"Set footfall to {footfall} for religious amenity {amenity_type}")

    # --- Country-wide amenities ---
    elif amenity_type in country_wide_amenities:
        # use total population and total number of amenities
        country_amenity_matches = amenity_count_whole_country.loc[
            amenity_count_whole_country['amenity_type'] == amenity_type, 'count'
        ].values
        
        if len(country_amenity_matches) == 0:
            print(f"Warning: No count data found for country-wide amenity {amenity_type}. Skipping.")
            continue
            
        amenity_count = country_amenity_matches[0]
        footfall = (total_population / amenity_count) * frequency_country_wide
        
        gdf.at[idx, 'footfall'] = footfall
        print(f"Set footfall to {footfall} for country-wide amenity {amenity_type}")
    
    elif amenity_type in tourism_amenities:# https://www.singstat.gov.sg/find-data/search-by-theme/industry/tourism/latest-data
        # use tourism numbers
        annual_tourism_numbers = 165263000
        weekly_tourism_numbers = annual_tourism_numbers / 52 # number of new tourist arrivals in a week
        
        # get number of hotels 
        amenity_matches = amenity_count_whole_country.loc[
            amenity_count_whole_country['amenity_type'] == amenity_type, 'count'
        ].values
        if amenity_type == 'hotels':
            # divide by number of hotels
            footfall = (weekly_tourism_numbers / amenity_matches[0])*3.56 # assume average stay is 3.56 days
            gdf.at[idx, 'footfall'] = footfall
            print(f"Set footfall to {footfall} for tourism amenity HOTELS")
        else:
            footfall = weekly_tourism_numbers
            gdf.at[idx, 'footfall'] = footfall
            print(f"Set footfall to {footfall} for tourism amenity {amenity_type}") # assumes that each tourist visits attraction once 
    
    elif amenity_type == by_demographic:
        # use special education enrolment numbers
        special_education_enrolment = 7818 # https://www.moe.gov.sg/news/parliamentary-replies/20240807-profile-of-students-enrolled-in-special-education-schools-and-publication-of-such-data
        # get number of special education schools
        special_education_matches = amenity_count_whole_country.loc[
            amenity_count_whole_country['amenity_type'] == amenity_type, 'count'
        ].values
        if len(special_education_matches) == 0:
            print(f"Warning: No count data found for special education amenity {amenity_type}. Skipping.")
            continue
            
        amenity_count = special_education_matches[0]
        footfall = (special_education_enrolment / amenity_count) * frequency_special_needs
        
        gdf.at[idx, 'footfall'] = footfall
        print(f"Set footfall to {footfall} for special education amenity {amenity_type}")
    
    else:
        print(f"Amenity type {amenity_type} not categorized for footfall calculation.")
        continue

Processing bus_depots in BISHAN
Set footfall to 611240.0 for bus_depots in BISHAN
Processing bus_depots in ANG MO KIO
Set footfall to 567980.0 for bus_depots in ANG MO KIO
Processing bus_depots in ANG MO KIO
Set footfall to 567980.0 for bus_depots in ANG MO KIO
Processing bus_depots in HOUGANG
Set footfall to 567980.0 for bus_depots in HOUGANG
Processing bus_depots in TAMPINES
Set footfall to 567980.0 for bus_depots in TAMPINES
Processing bus_depots in BUKIT BATOK
Set footfall to 1106210.0 for bus_depots in BUKIT BATOK
Processing bus_depots in TOA PAYOH
Set footfall to 1106210.0 for bus_depots in TOA PAYOH
Processing bus_depots in JURONG WEST
Set footfall to 1106210.0 for bus_depots in JURONG WEST
Processing bus_depots in QUEENSTOWN
Set footfall to 1106210.0 for bus_depots in QUEENSTOWN
Processing bus_depots in HOUGANG
Set footfall to 553105.0 for bus_depots in HOUGANG
Processing bus_depots in SUNGEI KADUT
Set footfall to 553105.0 for bus_depots in SUNGEI KADUT
Processing bus_depots in

In [60]:
gdf.head(25)

,amenity_id,amenity_type,planning_area,footfall,is_heartland
0,8d39567d-5ea6-5751-beec-88770a9be781,bus_depots,BISHAN,6.112400e+05,True
1,fbed27ca-8066-5e40-8cc5-d62cadfb738c,bus_depots,ANG MO KIO,5.679800e+05,True
2,fb09f377-194b-5c85-9942-cc74e9136575,bus_depots,ANG MO KIO,5.679800e+05,True
3,187dda5f-1316-5f06-854b-0718ea11f55c,bus_depots,HOUGANG,5.679800e+05,True
4,eb89bf2e-7583-5961-95b3-2c647513f432,bus_depots,TAMPINES,5.679800e+05,True
5,9f90d676-aa68-561b-b7df-d4197af2cfec,bus_depots,BUKIT BATOK,1.106210e+06,True
6,c2d042e1-f588-5613-94c6-c8b8df7b97e2,bus_depots,TOA PAYOH,1.106210e+06,True
7,b4a4336a-616a-51ff-84a1-38c798ea10b6,bus_depots,JURONG WEST,1.106210e+06,True
8,5ce8041a-bab4-5e2f-9b5f-2270b24bdac4,bus_depots,QUEENSTOWN,1.106210e+06,True
9,5f7c06af-f531-5c81-87af-44da616a0ee8,bus_depots,HOUGANG,5.531050e+05,True


In [61]:
# Convert to GeoDataFrame
gdf.to_csv("revised_amenity_footfall_data.csv", index=False)

